# Q01 — Rows With Multiple Missing Values: Boolean Indexing and Filtering

**Python for Data Science — Company Interview Question Bank**

| Field | Detail |
|---|---|
| Question ID | `PANDAS-02-Q001` |
| Primary category | **02 — Indexing, Selection and Filtering** |
| Reported company | **Google / YouTube** (as attributed by StrataScratch) |
| Associated role | Data Analyst; relevant to Data Scientist and Quant data-cleaning workflows |
| Source question | **Rows With Missing Values**, StrataScratch **ID 2106** |
| Original format | **SQL**; this notebook is an **adaptation to Python/Pandas** |
| Published difficulty | Medium |
| Extended depth | Advanced–Expert (original educational follow-ups) |
| Key topics | Boolean masks, `.loc`, row-wise reduction, missing-value semantics, index preservation, label alignment, complexity |
| Source | [StrataScratch — Rows With Missing Values](https://platform.stratascratch.com/coding/2106-rows-with-missing-values) |

> **Source and attribution note:** StrataScratch lists this as a Google-associated question involving YouTube's `user_flags` table. Its published formulation is a SQL task, **not a verified verbatim Pandas interview question**. The concrete sample records, Pandas implementation, edge cases, tests, and advanced follow-ups below are educational adaptations; do not attribute them to Google's original interview.

## 1. Interview Question

**Business context (paraphrased from the published problem).** The data engineering team needs to inspect records in a video-flagging dataset that have **missing values in more than one field**.

**Pandas adaptation:** Given a DataFrame `user_flags` with columns

- `user_firstname`
- `user_lastname`
- `video_id`
- `flag_id`

return **every row with at least two missing values** across those four fields.

### Requirements

1. Determine missingness **per row**, not per column.
2. Keep rows where the number of missing values is **≥ 2** (including exactly two).
3. Preserve the original **row index**, row order, and column order.
4. Do not modify the input DataFrame.
5. Use vectorized Pandas operations, not an explicit Python loop over rows.

**Assumptions:** The four required column names are unique. Unless otherwise stated, Pandas' native missing-value rules apply; blank strings are not automatically treated as missing.

## 2. Mathematical Definition

Let a table contain $n$ rows and $k=4$ inspected columns. Define

$$
I_{ij} =
\begin{cases}
1, & \text{if value } x_{ij} \text{ is missing}, \\
0, & \text{otherwise}.
\end{cases}
$$

For each row $i$, the number of missing fields is

$$
m_i = \sum_{j=1}^{k} I_{ij}.
$$

The row-selection predicate is

$$
\operatorname{keep}(i) = \mathbf{1}_{\{m_i \geq 2\}}.
$$

The inequality is **inclusive**. Rows with exactly two missing values must be returned.

## 3. Setup and Sample Data

The input below is **synthetic**, constructed to test both the core rule and important boundary cases. A nonconsecutive index makes index preservation observable. Rows include `None`, `np.nan`, `pd.NA`, fully missing records, and strings consisting only of spaces.

In [1]:
import numpy as np
import pandas as pd

print("Pandas version:", pd.__version__)

FIELDS = ["user_firstname", "user_lastname", "video_id", "flag_id"]

user_flags = pd.DataFrame(
    {
        "user_firstname": ["Richard", "Courtney", "Gina", "Greg", None, "Avery", None, None, " "],
        "user_lastname": ["Hasson", None, "Korman", None, None, "Park", None, None, "Lopez"],
        "video_id": ["v001", "v002", pd.NA, "v004", "v005", "v006", None, None, "v009"],
        "flag_id": ["f001", np.nan, None, None, "f005", None, "f007", None, "f009"],
    },
    index=pd.Index([101, 102, 105, 108, 110, 115, 120, 125, 130], name="record_id"),
)

print(user_flags.to_string())
assert user_flags.columns.tolist() == FIELDS
assert user_flags.index.tolist() == [101, 102, 105, 108, 110, 115, 120, 125, 130]

Pandas version: 2.3.3
          user_firstname user_lastname video_id flag_id
record_id                                              
101              Richard        Hasson     v001    f001
102             Courtney          None     v002     NaN
105                 Gina        Korman     <NA>    None
108                 Greg          None     v004    None
110                 None          None     v005    f005
115                Avery          Park     v006    None
120                 None          None     None    f007
125                 None          None     None    None
130                              Lopez     v009    f009


## 4. Standard Pandas Solution

The key expression is `df[fields].isna().sum(axis=1) >= 2`.

- `df.loc[:, fields]`: selects the **four inspected columns** by label.
- `.isna()`: returns a same-shaped Boolean DataFrame (`True` for missing values).
- `.sum(axis=1)`: counts `True` values **across columns** for each row.
- `.ge(2)`: produces a Boolean Series identifying rows to retain.
- `.loc[mask]`: filters rows **by label-aware Boolean indexing**; selected rows keep their original index and order.
- `.copy()`: returns an explicit independent table for ordinary column value edits.

In [2]:
missing_count = user_flags[FIELDS].isna().sum(axis=1)
keep_mask = missing_count.ge(2)
result = user_flags.loc[keep_mask].copy()

result = user_flags.loc[user_flags[FIELDS].isna().sum(axis=1) >= 2].copy()

print("Missing values per row:")
print(missing_count.to_string())
print("\nBoolean selection mask:")
print(keep_mask.to_string())
print("\nFiltered records:")
print(result.to_string())

Missing values per row:
record_id
101    0
102    2
105    2
108    2
110    2
115    1
120    3
125    4
130    0

Boolean selection mask:
record_id
101    False
102     True
105     True
108     True
110     True
115    False
120     True
125     True
130    False

Filtered records:
          user_firstname user_lastname video_id flag_id
record_id                                              
102             Courtney          None     v002     NaN
105                 Gina        Korman     <NA>    None
108                 Greg          None     v004    None
110                 None          None     v005    f005
120                 None          None     None    f007
125                 None          None     None    None


## 5. Expected Output

**Selected record IDs:** `102`, `105`, `108`, `110`, `120`, `125`.

| record_id | user_firstname | user_lastname | video_id | flag_id | Missing values |
|---:|---|---|---|---|---:|
| 102 | Courtney | *missing* | v002 | *missing* | 2 |
| 105 | Gina | Korman | *missing* | *missing* | 2 |
| 108 | Greg | *missing* | v004 | *missing* | 2 |
| 110 | *missing* | *missing* | v005 | f005 | 2 |
| 120 | *missing* | *missing* | *missing* | f007 | 3 |
| 125 | *missing* | *missing* | *missing* | *missing* | 4 |

Rows `101` and `130` have no native missing values, while `115` has only one. The string `" "` in row `130` is **not** a missing value under `isna()`.

## 6. Interview-Ready Function

A reusable implementation explicitly checks the required schema and rejects duplicated column labels, which could make the target-column selection ambiguous.

It returns **all original columns**, as the published SQL task asks for matching rows rather than a subset of fields; the four specified fields determine the row predicate.

In [3]:
def find_rows_with_multiple_missing_values(
    frame: pd.DataFrame,
    columns: tuple[str, ...] = tuple(FIELDS),
    min_missing: int = 2,
) -> pd.DataFrame:
    """Return rows with at least `min_missing` nulls in `columns`.
    
    Input row labels, row ordering, and output column ordering are retained.
    """
    if not frame.columns.is_unique:
        raise ValueError("Column labels must be unique")
        
    if len(columns) == 0 or len(set(columns)) != len(columns):
        raise ValueError("Provide a nonempty collection of unique columns")
        
    absent = [column for column in columns if column not in frame.columns]
    if absent:
        raise KeyError(f"Missing required columns: {absent}")
        
    if not isinstance(min_missing, int) or isinstance(min_missing, bool):
        raise TypeError("min_missing must be an integer")
        
    if not 1 <= min_missing <= len(columns):
        raise ValueError("min_missing must be between 1 and the number of selected columns")

    null_counts = frame.loc[:, list(columns)].isna().sum(axis="columns")
    return frame.loc[null_counts.ge(min_missing)].copy()

filtered = find_rows_with_multiple_missing_values(user_flags)
print(filtered.to_string())

          user_firstname user_lastname video_id flag_id
record_id                                              
102             Courtney          None     v002     NaN
105                 Gina        Korman     <NA>    None
108                 Greg          None     v004    None
110                 None          None     v005    f005
120                 None          None     None    f007
125                 None          None     None    None


## 7. Automated Verification

A strong solution validates the actual selected labels, the schema, the nonmutation guarantee, and the difference between **exactly two** and **at least two** missing values.

In [4]:
expected_ids = [102, 105, 108, 110, 120, 125]
expected = user_flags.loc[expected_ids].copy()

pd.testing.assert_frame_equal(filtered, expected)
pd.testing.assert_frame_equal(result, expected)

assert filtered.index.tolist() == expected_ids
assert filtered.columns.tolist() == FIELDS
assert filtered.index.name == "record_id"

assert (
    user_flags.loc[expected_ids, FIELDS]
        .isna()
        .sum(axis=1) >= 2
).all()

assert (
    user_flags.loc[[101, 115, 130], FIELDS]
        .isna()
        .sum(axis=1) < 2
).all()

original_snapshot = user_flags.copy(deep=True)
_ = find_rows_with_multiple_missing_values(user_flags)
pd.testing.assert_frame_equal(user_flags, original_snapshot)
print("All core tests passed.")

All core tests passed.


## 8. Indexing Semantics — `.loc`, `.iloc`, and Alignment

### 8.1 Why `.loc[mask]`?

The Boolean Series produced by `isna().sum(axis=1).ge(2)` has the **same row index** as the original table. `.loc` uses row labels; selecting rows does **not** automatically reset or renumber the index.

### 8.2 Why not `.iloc[mask]`?

`.iloc` is **position-based** and does not accept an index-bearing Boolean Series as its row indexer. If positional selection is intentional, convert the mask to a Boolean NumPy array using `.to_numpy()`.

### 8.3 A meaningful alignment test

With **unique row labels**, an intentionally reordered Boolean mask can still be aligned by `.loc` to the corresponding row labels. Applying that same reversed mask as a positional array would pair mask positions with the wrong observations.

In [5]:
# Label-aware selection: reversed mask order does not change which labels match.
reversed_mask = keep_mask.iloc[::-1]
label_selected = user_flags.loc[reversed_mask]
pd.testing.assert_frame_equal(label_selected, expected)

# Correct position-based equivalent: keep the ORIGINAL mask ordering.
position_selected = user_flags.iloc[keep_mask.to_numpy()]
pd.testing.assert_frame_equal(position_selected, expected)

# A reversed *array* applies positionally and is NOT equivalent to label alignment.
wrong_positional = user_flags.iloc[reversed_mask.to_numpy()]
assert wrong_positional.index.tolist() != expected_ids

print("Label-aware .loc selection:", label_selected.index.tolist())
print("Positionally reversed mask:", wrong_positional.index.tolist())

Label-aware .loc selection: [102, 105, 108, 110, 120, 125]
Positionally reversed mask: [102, 105, 110, 115, 120, 125]


## 9. Alternative Implementations

### 9.1 Compact, idiomatic expression

Use this in a coding interview when the schema is trusted and one expression is preferable to a helper function.

In [6]:
concise = user_flags.loc[user_flags[FIELDS].isna().sum(axis=1).ge(2)]
pd.testing.assert_frame_equal(concise, expected)
print(concise.to_string())

          user_firstname user_lastname video_id flag_id
record_id                                              
102             Courtney          None     v002     NaN
105                 Gina        Korman     <NA>    None
108                 Greg          None     v004    None
110                 None          None     v005    f005
120                 None          None     None    f007
125                 None          None     None    None


### 9.2 Threshold via `notna()`

For exactly \(k\) inspected columns, the condition "at least \(t\) missing" is equivalent to "at most \(k-t\) nonmissing". This is mathematically equivalent but often less readable.

In [7]:
k = len(FIELDS)
max_present = k - 2
alternative = user_flags.loc[user_flags[FIELDS].notna().sum(axis=1).le(max_present)]
pd.testing.assert_frame_equal(alternative, expected)
print("notna()-based implementation matches the standard solution.")

notna()-based implementation matches the standard solution.


## 10. Computational Complexity and Memory

Let \(n\) be the number of rows, \(k\) the number of inspected columns, \(p\) the number of returned columns, and \(r\) the number of matching rows.

| Operation | Typical time cost | Temporary / output memory |
|---|---|---|
| Select `k` columns | Indexing-dependent; often cheap relative to scanning | May or may not share buffers |
| Compute `isna()` for inspected data | \(O(nk)\) | \(O(nk)\) Boolean matrix |
| Sum missing flags by row | \(O(nk)\) | \(O(n)\) row counts |
| Compare counts to threshold | \(O(n)\) | \(O(n)\) Boolean mask |
| Filter and explicitly copy output | \(O(rp)\), ignoring dtype-specific copying overhead | \(O(rp)\) for fixed-width columns |

A useful overall model for fixed-width data is

\[
T(n,k,p,r)=O(nk+rp), \qquad
M(n,k,p,r)=O(nk+rp).
\]

When \(k\) and \(p\) are fixed constants, this is **linear in row count**. These are asymptotic models rather than guaranteed exact costs: extension arrays, variable-length strings, index operations, and object-valued columns have their own storage/copy behavior. `.copy(deep=True)` does not recursively clone arbitrary nested Python objects.

## 11. Edge Cases and Data-Quality Policy

1. **Exactly two nulls:** must be included because the condition is `>= 2`, not `> 2`.
2. **All four nulls:** included.
3. **Zero or one null:** excluded.
4. **`None`, `np.nan`, `pd.NA`, `pd.NaT`:** treated as missing by the relevant Pandas missingness APIs.
5. **Empty strings, spaces, and literal text such as `"NULL"`:** not missing unless standardized first.
6. **Empty DataFrame with a valid schema:** return an empty DataFrame retaining columns and index metadata.
7. **Extra columns:** return them too, but only count missingness in the four specified columns.
8. **Duplicate row labels:** a row-wise Boolean mask created on the same DataFrame remains usable; do not confuse labels with unique record identifiers.
9. **Duplicate column labels / absent required columns:** the reusable function rejects these ambiguous or invalid schemas.
10. **Input mutation:** avoid inplace editing of the source or its potentially shared underlying objects.

In [8]:
# 1) Empty input retains schema and index metadata.
empty_output = find_rows_with_multiple_missing_values(user_flags.iloc[:0])
assert empty_output.empty
assert empty_output.columns.equals(user_flags.columns)
assert empty_output.index.name == "record_id"

# 2) Extra fields do not affect the missing-value threshold.
with_extra = user_flags.assign(source_system=[None] * len(user_flags))
extra_output = find_rows_with_multiple_missing_values(with_extra)
assert extra_output.index.tolist() == expected_ids
assert "source_system" in extra_output.columns

# 3) Blank strings and a text sentinel do not count as nulls automatically.
sentinels = pd.DataFrame(
    {
        "user_firstname": ["", "NULL", None],
        "user_lastname": [" ", "Lee", "Li"],
        "video_id": ["v1", "v2", None],
        "flag_id": [None, None, "f3"],
    },
    index=[1, 2, 3],
)
assert find_rows_with_multiple_missing_values(sentinels).index.tolist() == [3]

# 4) Repeated row labels do not change the row-level predicate.
nonunique_index = user_flags.copy()
nonunique_index.index = [0, 0, 1, 1, 2, 2, 3, 3, 4]
nonunique_output = find_rows_with_multiple_missing_values(nonunique_index)
assert len(nonunique_output) == len(expected_ids)

# 5) Validate schema errors explicitly.
try:
    find_rows_with_multiple_missing_values(user_flags.drop(columns=["flag_id"]))
except KeyError:
    pass
else:
    raise AssertionError("Missing required column should fail")

print("All edge-case tests passed.")

All edge-case tests passed.


## 12. Advanced Follow-up Questions — Educational Extensions

**F1. How do `.loc` and `.iloc` differ in this task?**  
`.loc` selects by labels and supports index-aligned Boolean Series; `.iloc` selects by integer position and requires a positional Boolean array for this use case.

**F2. What does `axis=1` mean in `sum(axis=1)`?**  
Reduce **across columns**, yielding one count per row. `axis=0` would produce one count per column and would not satisfy the task.

**F3. Why not use `dropna(thresh=...)` directly?**  
`dropna(thresh=...)` retains rows with at least a specified **number of nonmissing values**. For four inspected columns and at least two missing values, one can use `dropna(axis=0, thresh=1)` or `thresh=2` only with care: neither alone expresses **at most two present fields**. An explicit mask is clearer. A `dropna` approach would require constructing a suitable complement of the retained set or counting nonmissing values directly.

**F4. What if the actual business definition also treats blank strings as missing?**  
First standardize strings in the **target columns**, for example with an appropriate whitespace-trimming and empty-string-to-NA policy, then apply the same mask. Do not globally replace valid empty-string values without confirming the domain policy.

**F5. What if missingness in an unrelated metadata column should be ignored?**  
Compute `isna()` only over `FIELDS`, then apply the resulting mask to the **full table**.

**F6. Why might boolean indexing be misleading when row labels are duplicated?**  
A row label may not uniquely identify one event. Boolean filtering still works for a mask from the same table, but subsequent joins and `.loc[label]` can return multiple records; a unique event ID is often preferable.

**F7. Could this operation be applied to a MultiIndex or financial panel dataset?**  
Yes. The row-wise missingness predicate is independent of row-label structure, provided the DataFrame's column selection is well defined. Keeping the index can be crucial for instrument/time alignment.

**F8. Would applying `df.isna().sum(axis=1)` always be correct?**  
Only if **every** DataFrame column is meant to be checked. For the stated task, the denominator/target population is the four specified fields; extra columns should not change the answer.

## 13. Analytical Interpretation

This is a **data-quality screening** operation, not an imputation or deletion policy. It identifies records that may require investigation before downstream aggregation, modeling, or joins.

- **In product analytics:** multiple missing user/video identifiers can indicate upstream logging defects, failed instrumentation, or corrupted ingestion; the output alone does **not** prove which failure occurred.
- **In data science:** filtering out these records can cause selection bias if missingness is correlated with the target, segment, or measurement process.
- **In quantitative finance:** retaining the original index is essential when observations are labeled by security, trading session, timestamp, or a composite key. Incorrect alignment can silently corrupt feature/return mapping.
- **In production ETL:** monitoring the fraction of rows violating this threshold over time may provide a useful data-quality signal. Thresholds must be chosen in relation to business-critical fields, not merely by column count.

A senior-level explanation should explicitly distinguish the **selection predicate**, the **meaning of missingness**, **index alignment**, **output preservation**, and the **statistical consequences of dropping records**.

## 14. Concise Interview-Ready Answer

> First, restrict the missing-value check to the four relevant columns. `isna()` produces a Boolean matrix, `sum(axis=1)` counts missing fields for each row, and `ge(2)` gives the required row mask. I then use `.loc[mask]` to return those records while preserving their index, order, and all original columns. The operation is vectorized, runs in approximately linear time for a fixed schema, and avoids mutating the input.

```python
result = user_flags.loc[
    user_flags[["user_firstname", "user_lastname", "video_id", "flag_id"]]
    .isna()
    .sum(axis=1)
    .ge(2)
]
```

**Completion checklist:** Predict which rows survive; explain the role of `axis=1`; contrast `.loc` with `.iloc`; identify the blank-string edge case; and state when label preservation matters in a Data Science or Quant pipeline.